# Project 1
## 1. Incremental ingestion

In [116]:
#imports 
import os
import json
from datetime import datetime
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

In [117]:
spark = (
    SparkSession.builder
    .appName("Project1_Group_H")
    .enableHiveSupport()
    .config("spark.sql.autoBroadcastJoinThreshold", "10MB")
    .getOrCreate()
)

In [118]:
# ---- Paths ----
base_path = "/home/jovyan/work"

inbox_path = os.path.join(base_path, "data/inbox")
manifest_path = os.path.join(base_path, "state/manifest.json")
outbox_path = os.path.join(base_path, "data/outbox")
output_path = os.path.join(outbox_path, "trips_enriched.parquet")
summary_path = os.path.join(outbox_path, "monthly_borough_summary.parquet")

# ---- Create required directories ----
os.makedirs(f"{base_path}/state", exist_ok=True)
os.makedirs(outbox_path, exist_ok=True)

# ---- Load manifest if exists ----
if os.path.exists(manifest_path):
    with open(manifest_path, "r") as f:
        manifest = json.load(f)

else:
    manifest = {
        "trip_files": []
    }

In [119]:
# ---- Load TAXI ZONE LOOKUP every run ----
lookup_path = os.path.join(inbox_path, "taxi_zone_lookup.parquet")

if not os.path.exists(lookup_path):
    raise FileNotFoundError(f"Lookup file not found: {lookup_path}")

zones_raw = spark.read.parquet(lookup_path).cache()
print("Lookup rows:", zones_raw.count())

# ---- Get all TAXI TRIP files ----
all_files = [f for f in os.listdir(inbox_path) if f.endswith(".parquet") and f != "taxi_zone_lookup.parquet"]

# ---- Select only new taxi trip files (checks file size and name) ----
processed_index = {
    f["filename"]: f["file_size"]
    for f in manifest["trip_files"]
}

new_files = []
new_file_metadata = []

for file in all_files:
    full_path = os.path.join(inbox_path, file)
    current_size = os.path.getsize(full_path)

    if file not in processed_index or processed_index[file] != current_size:
        new_files.append(file)
        new_file_metadata.append({
            "filename": file,
            "file_size": current_size,
            "processed_at": datetime.now().isoformat()
        })

has_new_files = len(new_files) > 0

print("All trip files found:", all_files)
print("New files to process:", new_files)

Lookup rows: 265
All trip files found: ['yellow_tripdata_2025-02.parquet', 'yellow_tripdata_2025-01.parquet', 'yellow_tripdata_2024-12.parquet']
New files to process: []


In [120]:
# ---- Read new files ----
if not has_new_files:
    print("No new files found. Outputs and manifest remain unchanged.")
else:
    full_paths = [os.path.join(inbox_path, f) for f in new_files]
    run_ingested_at = datetime.now().isoformat(timespec="seconds")

    df_raw = spark.read.parquet(*full_paths)
    df_raw = (df_raw.withColumn("source_file", F.input_file_name())
            .withColumn("ingested_at", F.current_timestamp()))
    total_rows = df_raw.count()
    print("Total rows read:", total_rows)

No new files found. Outputs and manifest remain unchanged.


## 2. Transformations
### Parse and cast types

In [121]:
if has_new_files:
    df_casted = df_raw.select(
        F.col("VendorID").cast("int"),
        # timestamps
        F.col("tpep_pickup_datetime").cast("timestamp").alias("pickup_datetime"),
        F.col("tpep_dropoff_datetime").cast("timestamp").alias("dropoff_datetime"),
        
        # ints
        F.col("PULocationID").cast("int"),
        F.col("DOLocationID").cast("int"),
        F.col("passenger_count").cast("int"),

        # floats
        F.col("trip_distance").cast("double"),
        F.col("fare_amount").cast("double"),
        F.col("total_amount").cast("double"),

        #metadata
        F.col("source_file"),
        F.col("ingested_at")
    )

    df_casted.printSchema()
    df_casted.show(3, truncate=False)
else:
    print("Skipping cast step: no new files.")

Skipping cast step: no new files.


In [122]:
zones_raw.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


### Clean
Rules which we discussed during meetup.

| # | Rule | Column(s) | Condition kept |
|---|------|-----------|----------------|
| 1 | Non-null timestamps | `pickup_datetime`, `dropoff_datetime` | `isNotNull()` |
| 2 | Chronological trip | both timestamps | `dropoff > pickup` |
| 3 | Positive distance | `trip_distance` | `> 0` |
| 4 | Valid passenger count | `passenger_count` | `> 0` |
| 5 | Positive fare | `total_amount`, `fare_amount` | `> 0` |
| 6 | Valid location IDs | `PULocationID`, `DOLocationID` | `isNotNull()` |

In [123]:
if has_new_files:
    df_cleaned = df_casted.filter(
        F.col("pickup_datetime").isNotNull() &
        F.col("dropoff_datetime").isNotNull() &
        (F.col("dropoff_datetime") > F.col("pickup_datetime")) &
        (F.col("trip_distance") > 0) &
        (F.col("passenger_count") > 0) &
        ((F.col("total_amount") > 0) & (F.col("fare_amount") > 0)) &
        F.col("PULocationID").isNotNull() &
        F.col("DOLocationID").isNotNull()
    )

    raw_count = df_casted.count()
    after_clean_count = df_cleaned.count()

    print(f"Rows input: {raw_count}")
    print(f"Rows after clean: {after_clean_count} (removed {raw_count - after_clean_count})")
else:
    print("Skipping cleaning step: no new files.")

Skipping cleaning step: no new files.


In [124]:
if has_new_files:
    print("trip_distance <= 0:")
    df_casted.filter(
        F.col("trip_distance") <= 0
    ).select(
        "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count", "total_amount"
    ).show(2, truncate=False)

    print("passenger_count < 1:")
    df_casted.filter(
        F.col("passenger_count") < 1
    ).select(
        "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count", "total_amount"
    ).show(2, truncate=False)

    print("dropoff_datetime <= pickup_datetime:")
    df_casted.filter(
        F.col("pickup_datetime").isNotNull() &
        (F.col("dropoff_datetime") <= F.col("pickup_datetime"))
    ).select(
        "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count"
    ).show(1, truncate=False)
    df_casted.filter(
        F.col("pickup_datetime").isNotNull() &
        (F.col("dropoff_datetime") < F.col("pickup_datetime"))
    ).select(
        "pickup_datetime", "dropoff_datetime", "trip_distance", "passenger_count"
    ).show(1, truncate=False)
else:
    print("Skipping bad-row examples: no new files.")

Skipping bad-row examples: no new files.


### Deduplication
**Key:** `(pickup_datetime, dropoff_datetime, PULocationID, DOLocationID, trip_distance, vendorID)`  
Two real trips cannot share the same origin, destination, exact times, and distance.

In [125]:
deduplication_key = [
    "pickup_datetime",
    "dropoff_datetime",
    "PULocationID",
    "DOLocationID",
    "trip_distance",
    "VendorID",
]

if has_new_files:
    df_dedup = df_cleaned.dropDuplicates(deduplication_key)

    after_dedup_count = df_dedup.count()
    print(f"Rows after clean: {after_clean_count}")
    print(f"Rows after dedup: {after_dedup_count}(removed {after_clean_count - after_dedup_count} duplicates)")
else:
    print("Skipping dedup step: no new files.")

Skipping dedup step: no new files.


### Derived Columns


In [126]:
if has_new_files:
    df_derived = df_dedup.selectExpr(
        "*",
        "round((unix_timestamp(dropoff_datetime) - unix_timestamp(pickup_datetime)) / 60.0, 2) AS trip_duration_minutes",
        "to_date(pickup_datetime) AS pickup_date"
    )

    df_derived.select(
        "pickup_datetime", "dropoff_datetime",
        "trip_distance", "trip_duration_minutes", "pickup_date"
    ).show(5, truncate=False)
else:
    print("Skipping derived columns: no new files.")

Skipping derived columns: no new files.


## 3. Zone Enrichment
Cast column names for the zones once, then create separate pickup and dropoff views. the zone table has 265 rows therefore smaller than main data aand we'll use broacast join

In [127]:
if has_new_files:
    zones = zones_raw.select(
        F.col("LocationID").cast("int"),
        F.col("Zone").alias("zone_name"),
        F.col("Borough").alias("borough")
    )

    pickup_zones = zones.select(
        F.col("LocationID").alias("PULocationID"),
        F.col("zone_name").alias("pickup_zone"),
        F.col("borough").alias("pickup_borough")
    )

    dropoff_zones = zones.select(
        F.col("LocationID").alias("DOLocationID"),
        F.col("zone_name").alias("dropoff_zone"),
        F.col("borough").alias("dropoff_borough")
    )

    print("autoBroadcastJoinThreshold:", spark.conf.get("spark.sql.autoBroadcastJoinThreshold"))

    df_enriched = (
        df_derived
        .join(F.broadcast(pickup_zones), on="PULocationID", how="left")
        .join(F.broadcast(dropoff_zones), on="DOLocationID", how="left")
    )

    df_enriched.select(
        "pickup_datetime",
        "PULocationID", "pickup_zone", "pickup_borough",
        "DOLocationID", "dropoff_zone", "dropoff_borough"
    ).show(5, truncate=False)
else:
    print("Skipping enrichment step: no new files.")

Skipping enrichment step: no new files.


### End result

Required output fields must include at least:
* pickup and dropoff timestamps
* pickup and dropoff LocationID
* pickup and dropoff zone name (from lookup)
* passenger_count, trip_distance
* derived: trip_duration_minutes, pickup_date
* metadata: source_file, ingested_at

In [128]:
if has_new_files:
    df_output = df_enriched.select(
        "VendorID",
        "pickup_datetime",
        "dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "pickup_zone",
        "pickup_borough",
        "dropoff_zone",
        "dropoff_borough",
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "total_amount",
        "trip_duration_minutes",
        "pickup_date",
        F.date_format("pickup_date", "yyyy-MM").alias("pickup_month"),
        "source_file",
        "ingested_at"
    )

    df_output.printSchema()
    df_output.show(5, truncate=False)

    final_count = df_output.count()
    print(f"Input (raw): {raw_count}")
    print(f"After cleaning: {after_clean_count}")
    print(f"After dedup: {after_dedup_count}")
    print(f"Final output rows in this incremental batch: {final_count}")

    # unique months names come from the actual trip timestamps, not from filenames.
    # new_months = df_output.select("pickup_month").where(F.col("pickup_month").isNotNull()).distinct().cache()
    # print("Months found inside new data batch:")
    # new_months.orderBy("pickup_month").show(truncate=False)
else:
    print("Skipping final output projection: no new files.")

Skipping final output projection: no new files.


In [129]:
if has_new_files:
    df_output.explain(True)
else:
    print("Skipping physical plan: no new files.")

Skipping physical plan: no new files.


In [130]:
if has_new_files:
    # ---- Append enriched data to outbox parquet ----
    (
        df_output
        .repartition("pickup_month")
        .write
        .mode("append")
        .partitionBy("pickup_month")
        .parquet(output_path)
    )

    print(f"Appended {final_count} rows to {output_path}")
else:
    print("Skipping enriched output write: no new files.")

Skipping enriched output write: no new files.


In [131]:
if has_new_files:
   # ---- Incremental monthly borough summary ----
    summary_path = os.path.join(outbox_path, "monthly_borough_summary.parquet")

    new_months = df_output.select("pickup_month").distinct()

    if os.path.exists(summary_path):
        existing_months = spark.read.parquet(summary_path).select("pickup_month").distinct()
        months_to_process = new_months.join(existing_months, on="pickup_month", how="left_anti")
    else:
        months_to_process = new_months

    if months_to_process.limit(1).count() == 0:
        print("No new months to append in monthly_borough_summary.")
    else:
        df_monthly_borough_summary = (
            df_output
            .join(F.broadcast(months_to_process), on="pickup_month", how="inner")
            .groupBy("pickup_month", "pickup_borough")
            .agg(
                F.count("*").alias("trip_count"),
                F.round(F.sum("total_amount"), 2).alias("total_revenue"),
                F.round(F.avg("trip_distance"), 2).alias("avg_trip_distance"),
                F.round(F.avg("trip_duration_minutes"), 2).alias("avg_trip_duration_minutes")
            )
        )

        (
            df_monthly_borough_summary
            .repartition("pickup_month")
            .write
            .mode("append")
            .partitionBy("pickup_month")
            .parquet(summary_path)
        )

        print(f"Appended monthly borough summary to {summary_path}")
        existing_files = {
            f["filename"]: f
            for f in manifest.get("trip_files", [])
        }

        for meta in new_file_metadata:
            existing_files[meta["filename"]] = meta

        manifest["trip_files"] = list(existing_files.values())

        with open(manifest_path, "w", encoding="utf-8") as f:
            json.dump(manifest, f, indent=2)

        print(f"Manifest updated at {manifest_path}")
else:
    print("Skipping monthly summary + manifest update: no new files.")

Skipping monthly summary + manifest update: no new files.
